# 🧬 ARISE-V6 Component Ablation Suite (Kaggle GPU Runner)

This notebook runs systematic component ablations centered on **ARISE-V6 (ARISE + 2-Stage Consensus DEC)** as the base anchor across 4 architectural tracks (26 total variants):
1. **`fusion-ablation`** (`F0` - `F6`): Linear, Local-Global, Variance Weight, Graph-Masked Cross-Attn, Bilinear Tensor, Gated Multi-Modal, CAGE Bidirectional QKV Cross-Fusion
2. **`loss-ablation`** (`L0` - `L5`): Standard V6, Dense Relational, Sinkhorn OT, Spatial InfoNCE, Spatial Potts, Uncertainty Balancing
3. **`encoder-ablation`** (`E0` - `E5`): Standard GCN, Global Transformer, 3-Node Motif ($M_3$), 4-Node Cycle Motif ($M_4$), Heat Diffusion Wavelet, GATv2 Attention
4. **`contrastive-ablation`** (`C0` - `C6`): Base V6, GATCL Cross-Modal CL, Proust DGI Bilinear CL, Spatial Neighbor InfoNCE, Cluster-Prototype InfoNCE, Graph Dual-View Consistency, Hybrid Multi-Level CL

> ⚠️ **Kaggle Setup Reminder**:
> - In the right sidebar under **Settings** $\rightarrow$ **Internet**, turn **Internet ON**.
> - In the right sidebar under **Settings** $\rightarrow$ **Accelerator**, select **GPU T4 x2** or **GPU P100**.

### 📦 1. Clone Repository & Setup Working Directory

In [ ]:
import os

# Clone repo if not already present in working directory
if not os.path.exists('Arise-V6'):
    !git clone https://github.com/Asadullah-Imran/Arise-V6.git

%cd /kaggle/working/Arise-V6
!pwd

### ⚙️ 2. Install Required Dependencies & Verify CUDA

In [ ]:
!pip install -q scanpy anndata scikit-misc torch-geometric gdown

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

### 🧪 3. Run Sanity Unit Tests
Verifies architecture instantiation and forward pass across all 26 variants before launching benchmark experiments.

In [ ]:
!python test_ablation_modules.py

### 🚀 4. Execute Ablation Runs
Configure your run parameters below and execute.

In [ ]:
# Options:
# Track: 'fusion', 'loss', 'encoder', 'contrastive', or 'all'
# Variant: Specific variant name (e.g. 'F6_QKVCrossFusion') or 'all'
# Dataset: 'all', '0', '1', '2', '3', '4', '5', '0,1', or '2,3,4,5'

TRACK = "fusion"              # Options: "all", "fusion", "loss", "encoder", "contrastive"
VARIANT = "F6_QKVCrossFusion" # Options: "F6_QKVCrossFusion", "all", "F0_Linear", etc.
DATASET = "all"               # Options: "all", "0", "1", "2", "3", "4", "5", "0,1"
SEEDS = "42 1234 2024"        # 3 standard benchmark seeds
EPOCHS = 400
PRETRAIN_EPOCHS = 250
FINETUNE_EPOCHS = 150

!python Arise_V6_Unified_Runner.py \
    --track {TRACK} \
    --variant {VARIANT} \
    --dataset {DATASET} \
    --seeds {SEEDS} \
    --epochs {EPOCHS} \
    --pretrain_epochs {PRETRAIN_EPOCHS} \
    --finetune_epochs {FINETUNE_EPOCHS} \
    --device cuda \
    --out_dir results_v6_ablation

### 📊 5. View & Summarize Ablation Results

In [ ]:
import os
import pandas as pd

csv_path = 'results_v6_ablation/arise_v6_ablation_results.csv'
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f"Total runs recorded: {len(df)}")
    display(df)
    
    print("\n" + "="*50)
    print("🏆 Performance Summary (Mean ± Std):")
    print("="*50)
    summary = df.groupby(['track', 'variant'])[['ARI', 'NMI', 'Silhouette']].agg(['mean', 'std'])
    display(summary)
else:
    print("No results file found yet. Run an experiment in Step 4 first.")